# p_play term — P(play) = P(minutes > 0)

**Render-not-decide.** A re-runnable view of the `p_play` appearance gate, **not** the record. Logic is in `model/terms/p_play/` (+ the shared binary base).

A **per-position logistic** on `played = 1{minutes > 0}` — the probability a player features at all, *before* the `minutes` term's P(>=60' | played). Unlike p60, **all four positions** are fit by logistic (a real starter-vs-backup split for GK too), and the population + TRAIN keep the blank rows. `compose` (keep_all) multiplies P(play) onto the conditional decomposition. See `ASSUMPTIONS.md`.

## Setup

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display

from dal.pipeline import load as load_mart
from model.terms.p_play import PlayModel, PlayTerm

mart = load_mart()
term = PlayTerm(PlayModel(variant="selected"))
term.model.pool.minimal, [f.name for f in term.model.pool.candidates]

## Pre-fit assumptions
> Binary term: family is logistic by construction; the floor is class balance (appearance base rate) + enough blank events. See `ASSUMPTIONS.md` §1-2.

In [ ]:
report = term.model.check_assumptions(PlayModel.population(mart))
print(f"detectable={report.detectable}  n_train={report.n_train}")
report.dispersion  # base_rate / n_events / family

## Gate — P(play) vs the lagged minutes level (minutes_roll3)
> Does the per-position logistic rank *who features next* better than a raw lagged-minutes level? (Value is also a calibrated probability, read below.)

In [ ]:
gate = term.validate(mart)
display(gate.table)

ax = gate.table.set_index("position")[["baseline", "p_play"]].plot.bar(figsize=(7, 3.5))
ax.set_ylabel("within-position Spearman"); ax.set_title("P(play): model vs lagged-minutes baseline"); plt.tight_layout()

## Calibration view — is P(play) a good probability?
> Bin P(play) and compare to the realized appearance (`played`) rate — including the blank rows.

In [ ]:
fitted = term.model.fit(mart)
pop = PlayModel.population(mart).assign(p_play=fitted.predictions)
ev = pop[pop['p_play'].notna()].copy()
ev['bucket'] = (ev['p_play'] * 10).round() / 10
cal = ev.groupby('bucket').agg(predicted=('p_play', 'mean'), realized=('played', 'mean'), n=('played', 'size'))
display(cal)
ax = cal.plot(y=['predicted', 'realized'], marker='o', figsize=(5, 4), title='P(play) calibration')
ax.set_xlabel('P(play) bucket'); plt.tight_layout()